# Final evaluation of the LoRA models on the test sets

The LoRA run is selected on rand-dev (macro-F1): **L2** (no lemma). L1 (with lemma) is evaluated as well,
to answer the question of whether the lemma is useful. No training happens here:
the saved adapters are loaded and each test set is used once.

Inputs: the folders `L1_adapter` and `L2_adapter` must be attached to this notebook (Add Input).

In [1]:
%pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully un

In [2]:
from unsloth import FastLanguageModel

import gc, glob, os
import torch, transformers, peft, bitsandbytes
import pandas as pd

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
transformers 5.16.1 | peft 0.20.0 | bitsandbytes 0.50.2


In [3]:
# Shared code: identical to lab1_shared.py (used by both the ICL and LoRA parts)

import os
import re
import gzip
import random
import urllib.request

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report


# =====================================================================
# 1. Fixed settings -- any change must be made by both team members
# =====================================================================

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
SEED = 42
MAX_NEW_TOKENS = 10        # labels are short; 10 new tokens are enough

DEV_SUBSET_N = None        # None = full rand-dev; set an int (e.g. 300) to tune on a fixed subset
DROP_LABELS = []           # hypersenses to exclude (e.g. ["other"]), if we decide to

DATA_URL = ("http://www.linguist.univ-paris-diderot.fr/~mcandito/divers/"
            "super_wiktionary_labeled_data_simplified.noex.25labels.tsv.gz")

# Supersense -> hypersense mapping, copied from the lab notebook
SUPER2HYPER = {
    'artifact': 'inanimate',
    'body': 'inanimate',
    'food': 'inanimate',
    'object': 'inanimate',
    'plant': 'inanimate',
    'substance': 'inanimate',
    'act': 'dynamic_situation',
    'event': 'dynamic_situation',
    'phenomenon': 'dynamic_situation',
    'animal': 'animate',
    'person': 'animate',
    'groupxperson': 'animate',
    'cognition': 'information',
    'communication': 'information',
    'quantity': 'quantity',
    'attribute': 'stative_situation',
    'feeling': 'stative_situation',
    'relation': 'stative_situation',
    'state': 'stative_situation',
    'time': 'time',
    'possession': 'possession',
    'institution': 'institution',
    'act*cognition': 'dynamic_situation*information',
    'artifact*cognition': 'inanimate*information',
    'other': 'other',
}

# Field names of the input, depending on the prompt language
FIELDS = {
    "en": ("Lemma", "Definition"),
    "fr": ("Lemme", "Définition"),
}


def set_seed(seed=SEED):
    """Fix Python, NumPy and PyTorch random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


# =====================================================================
# 2. Data loading and preparation
# =====================================================================

def load_data(url=DATA_URL):
    """Download the gzipped TSV (if not already present) and load it."""
    filename = url.split('/')[-1]
    if not os.path.exists(filename):
        urllib.request.urlretrieve(url, filename)
    with gzip.open(filename, 'rt', encoding='utf-8') as f:
        return pd.read_csv(f, sep='\t')


def prepare_data(df):
    """Add the hypersense column, drop protect-frsemcor, split by the 'set' column.
    Returns (splits, labels):
      splits: dict with keys 'train', 'rand-dev', 'rand-test', 'freq-test'
      labels: sorted list of hypersense labels, shared by both methods"""
    df = df.copy()
    df['hypersense'] = df['supersense'].map(SUPER2HYPER)
    missing = df.loc[df['hypersense'].isna(), 'supersense'].unique()
    assert len(missing) == 0, f"Supersenses missing from SUPER2HYPER: {missing}"

    df = df[df['set'] != 'protect-frsemcor']
    if DROP_LABELS:
        df = df[~df['hypersense'].isin(DROP_LABELS)]

    # Sets other than the four below (if any) are not used
    print(df['set'].value_counts(), "\n")

    names = ['train', 'rand-dev', 'rand-test', 'freq-test']
    splits = {n: df[df['set'] == n].reset_index(drop=True) for n in names}
    if DEV_SUBSET_N:
        splits['rand-dev'] = (splits['rand-dev']
                              .sample(DEV_SUBSET_N, random_state=SEED)
                              .reset_index(drop=True))

    labels = sorted(df['hypersense'].unique())
    for n in names:
        print(f"{n:10s}: {len(splits[n])} senses")
        assert len(splits[n]) > 0, f"Split '{n}' is empty: check the values of the 'set' column"
    print("Labels:", labels)
    return splits, labels


# =====================================================================
# 3. Input format -- used by both ICL and LoRA
# =====================================================================

def format_input(row, lang="en", use_lemma=True):
    """Turn one word sense into the input text, e.g.
         Lemma: béruguette
         Definition: Variété d'olive allongée à pulpe ferme."""
    f_lemma, f_def = FIELDS[lang]
    text = f"{f_def}: {row['definition']}"
    if use_lemma:
        text = f"{f_lemma}: {row['lemma']}\n" + text
    return text


# =====================================================================
# 4. Batched generation -- used by both methods at inference time
# =====================================================================

def generate_batch(model, tokenizer, messages_list, batch_size=16):
    """messages_list: one chat (list of messages) per instance.
    Returns the raw generated strings, in the same order as messages_list.
    Greedy decoding, MAX_NEW_TOKENS, left padding, batches sorted by length."""
    import torch

    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"              # required for batched generation
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in messages_list]
    # Sort by length so each batch needs little padding; restore order at the end
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outputs = [None] * len(prompts)

    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        # The chat template already contains the special tokens
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt",
                        padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        texts = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                       skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outputs[i] = t.strip()

    tokenizer.padding_side = old_side
    return outputs


# =====================================================================
# 5. Parsing and evaluation
# =====================================================================

def parse_output(text, labels):
    """Extract a label from the generated text. Returns (prediction, status) with
    status 'valid' (exactly one label), 'none' (no label) or 'multiple' (several labels).
    Boundaries exclude letters, digits, '_' and '*', so 'animate' does not match
    inside 'inanimate', and 'inanimate' does not match inside 'inanimate*information'."""
    text = text.lower()
    found = [l for l in labels
             if re.search(r'(?<![\w*])' + re.escape(l.lower()) + r'(?![\w*])', text)]
    if len(found) == 1:
        return found[0], "valid"
    return "INVALID", ("none" if not found else "multiple")


def evaluate(rows_df, raw_outputs, labels, verbose=True):
    """Returns (metrics, res):
      metrics: accuracy and macro-F1 (invalid outputs count as errors), invalid rates
      res: one row per instance, for saving and error analysis"""
    parsed = [parse_output(t, labels) for t in raw_outputs]
    res = pd.DataFrame({
        "sense_id":   rows_df["sense_id"].values,
        "lemma":      rows_df["lemma"].values,
        "definition": rows_df["definition"].values,
        "gold":       rows_df["hypersense"].values,
        "raw":        raw_outputs,
        "pred":       [p for p, _ in parsed],
        "status":     [s for _, s in parsed],
    })
    valid = res["status"] == "valid"
    # Macro-F1 is averaged over the labels that occur in this split only,
    # so that a label absent from the split does not pull the average down
    present = [l for l in labels if l in set(res["gold"])]
    metrics = {
        "n":                len(res),
        "accuracy":         accuracy_score(res["gold"], res["pred"]),
        "macro_f1":         f1_score(res["gold"], res["pred"], labels=present,
                                     average="macro", zero_division=0),
        "invalid_rate":     1 - valid.mean(),
        "no_label_rate":    (res["status"] == "none").mean(),
        "multi_label_rate": (res["status"] == "multiple").mean(),
        "acc_on_valid":     accuracy_score(res.loc[valid, "gold"], res.loc[valid, "pred"])
                            if valid.any() else 0.0,
    }
    if verbose:
        for k, v in metrics.items():
            print(f"{k:18s}: {v:.4f}" if isinstance(v, float) else f"{k:18s}: {v}")
        print(classification_report(res["gold"], res["pred"], labels=labels, zero_division=0))
    return metrics, res


# =====================================================================
# 6. Saving results -- {run}_{split}_preds.csv + one shared results_summary.csv
# =====================================================================

def save_results(run_name, split_name, metrics, res, out_dir, config=None):
    """run_name: e.g. 'E1', 'L1'; split_name: 'rand-dev', 'rand-test' or 'freq-test';
    config: optional dict describing the run, e.g. {'method': 'ICL', 'lang': 'en', 'lemma': True}"""
    os.makedirs(out_dir, exist_ok=True)
    res.to_csv(os.path.join(out_dir, f"{run_name}_{split_name}_preds.csv"), index=False)

    row = {"run": run_name, "split": split_name, **(config or {}), **metrics}
    summary_path = os.path.join(out_dir, "results_summary.csv")
    summary = pd.DataFrame([row])
    if os.path.exists(summary_path):
        summary = pd.concat([pd.read_csv(summary_path), summary], ignore_index=True)
    summary.to_csv(summary_path, index=False)
    print(f"Saved: {run_name} / {split_name}")

In [4]:
set_seed()
splits, labels = prepare_data(load_data())

set
train                    10604
freq-dev                  1760
freq-test                 1521
protect-frsemcor-dev      1041
protect-frsemcor-test      985
rand-test                  678
rand-dev                   570
Name: count, dtype: int64 

train     : 10604 senses
rand-dev  : 570 senses
rand-test : 678 senses
freq-test : 1521 senses
Labels: ['animate', 'dynamic_situation', 'dynamic_situation*information', 'inanimate', 'inanimate*information', 'information', 'institution', 'other', 'possession', 'quantity', 'stative_situation', 'time']


## Settings
Each run must be evaluated with exactly the input format it was trained with.

In [5]:
RUNS = {
    "L2": {"lemma": False},    # selected on rand-dev
    "L1": {"lemma": True},     # same settings with the lemma, for the lemma comparison
}
LANG = "en"
MAX_SEQ_LEN = 512
OUT_DIR = "/kaggle/working"

# Locate the attached adapter folders (any folder under /kaggle/input containing adapter_config.json)
ADAPTERS = {os.path.basename(os.path.dirname(p)).replace("_adapter", ""): os.path.dirname(p)
            for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)}
print(ADAPTERS)
assert all(r in ADAPTERS for r in RUNS), "attach the L1_adapter and L2_adapter folders as input"

# Same instruction and chat format as in training
INSTRUCTION = {
    "en": "Classify this French word sense into a hypersense.",
    "fr": "Classe ce sens de mot dans une hypersense.",
}

def make_messages(row, use_lemma):
    return [{"role": "user",
             "content": INSTRUCTION[LANG] + "\n" + format_input(row, LANG, use_lemma)}]

{'L1': '/kaggle/input/datasets/joycelzzhang/lora-test-l1-l2/L1_adapter', 'L2': '/kaggle/input/datasets/joycelzzhang/lora-test-l1-l2/L2_adapter'}


## Majority-class baseline
Predicts the most frequent hypersense of the training set for every instance.

In [6]:
majority = splits["train"]["hypersense"].value_counts().idxmax()
print("majority class (train):", majority)
for split in ["rand-dev", "rand-test", "freq-test"]:
    rows = splits[split]
    metrics, res = evaluate(rows, [majority] * len(rows), labels, verbose=False)
    print(f"{split:10s} acc={metrics['accuracy']:.4f}  macro_F1={metrics['macro_f1']:.4f}")
    if split != "rand-dev":
        save_results("majority", split, metrics, res, OUT_DIR, {"method": "majority baseline"})

majority class (train): inanimate
rand-dev   acc=0.2386  macro_F1=0.0321
rand-test  acc=0.3156  macro_F1=0.0400
Saved: majority / rand-test
freq-test  acc=0.2091  macro_F1=0.0288
Saved: majority / freq-test


## Evaluation
For each adapter, rand-dev is evaluated first only as a sanity check: it should reproduce the score
obtained at the end of training, which confirms that the adapter and input format are loaded correctly.

In [7]:
for run, cfg in RUNS.items():
    print(f"\n========== {run} (lemma={cfg['lemma']}) ==========")
    # Loading the adapter folder loads the base model and attaches the trained LoRA weights
    model, tokenizer = FastLanguageModel.from_pretrained(
        ADAPTERS[run], max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
    FastLanguageModel.for_inference(model)

    for split in ["rand-dev", "rand-test", "freq-test"]:
        rows = splits[split]
        raw = generate_batch(model, tokenizer,
                             [make_messages(r, cfg["lemma"]) for _, r in rows.iterrows()],
                             batch_size=32)
        print(f"--- {split} ---")
        metrics, res = evaluate(rows, raw, labels, verbose=(split != "rand-dev"))
        if split == "rand-dev":
            print(f"sanity check, rand-dev macro-F1 = {metrics['macro_f1']:.4f}")
        else:
            save_results(run, split, metrics, res, OUT_DIR,
                         {"method": "LoRA", "lang": LANG, "lemma": cfg["lemma"],
                          "modules": "attn+mlp", "output": "generation"})

    # Free GPU memory before loading the next adapter
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


========== L2 (lemma=False) ==========
==((====))==  Unsloth 2026.10.7: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.10.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more in

...

--- rand-dev ---
sanity check, rand-dev macro-F1 = 0.6741


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

--- rand-test ---
n                 : 678
accuracy          : 0.8673
macro_f1          : 0.7304
invalid_rate      : 0.0000
no_label_rate     : 0.0000
multi_label_rate  : 0.0000
acc_on_valid      : 0.8673
                               precision    recall  f1-score   support

                      animate       0.97      0.99      0.98       218
            dynamic_situation       0.91      0.78      0.84        77
dynamic_situation*information       0.40      0.67      0.50         3
                    inanimate       0.90      0.91      0.90       214
        inanimate*information       0.78      0.82      0.80        17
                  information       0.68      0.76      0.72        45
                  institution       0.70      1.00      0.82         7
                        other       0.33      0.12      0.17        26
                   possession       0.60      1.00      0.75         3
                     quantity       0.53      0.57      0.55        14
            st

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

--- freq-test ---
n                 : 1521
accuracy          : 0.7567
macro_f1          : 0.6958
invalid_rate      : 0.0000
no_label_rate     : 0.0000
multi_label_rate  : 0.0000
acc_on_valid      : 0.7567
                               precision    recall  f1-score   support

                      animate       0.97      0.84      0.90       128
            dynamic_situation       0.84      0.84      0.84       354
dynamic_situation*information       0.52      0.52      0.52        46
                    inanimate       0.80      0.91      0.85       318
        inanimate*information       0.72      0.70      0.71        37
                  information       0.67      0.74      0.70       163
                  institution       0.39      0.93      0.55        28
                        other       0.70      0.23      0.34       163
                   possession       0.71      0.77      0.74        22
                     quantity       0.77      0.49      0.60        47
            s

In [8]:
summary = pd.read_csv(f"{OUT_DIR}/results_summary.csv")
summary[["run", "split", "lemma", "accuracy", "macro_f1", "invalid_rate"]].round(4)

,run,split,lemma,accuracy,macro_f1,invalid_rate
0,majority,rand-test,NaN,0.3156,0.0400,0.0
1,majority,freq-test,NaN,0.2091,0.0288,0.0
2,L2,rand-test,False,0.8540,0.7169,0.0
3,L2,freq-test,False,0.7186,0.6458,0.0
4,L1,rand-test,True,0.8673,0.7304,0.0
5,L1,freq-test,True,0.7567,0.6958,0.0
